In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import glob
import os

# Ensure tabulate is available for table formatting
try:
    from tabulate import tabulate
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tabulate"])
    from tabulate import tabulate

def process_biomechanics_data(folder_path):
    file_pattern = os.path.join(folder_path, "*.xlsx")
    all_files = glob.glob(file_pattern)
    files = [f for f in all_files if not os.path.basename(f).startswith('~$')]
    
    if not files:
        print(f"No valid Excel files found in: {folder_path}")
        return

    # Containers for data
    time_wise_data = []
    angle_stats = []
    kinematic_stats = [] 
    
    for file in files:
        try:
            # Read and clean data
            df = pd.read_excel(file, skiprows=10, engine='openpyxl')
            df = df.iloc[:-6].reset_index(drop=True)
            df.columns = df.columns.astype(str).str.strip().str.lower()
            
            target_time, target_angle = 'time', 'left knee'
            if target_time not in df.columns or target_angle not in df.columns:
                print(f"Skipping {file}: Required columns not found.")
                continue

            time = df[target_time].values
            angle = df[target_angle].values
            
            # Calculate Kinematic Derivatives
            vel = np.gradient(angle, time)
            acc = np.gradient(vel, time)
            jerk = np.gradient(acc, time)
            
            video_name = os.path.splitext(os.path.basename(file))[0]

            # 1. Create Time-Wise Data Frame for this file
            file_df = pd.DataFrame({
                'Video_ID': video_name,
                'Time_sec': time,
                'Angle_deg': angle,
                'Angular_Vel': vel,
                'Angular_Acc': acc,
                'Angular_Jerk': jerk
            })
            time_wise_data.append(file_df)
            
            # 2. Calculate Descriptive Statistics for Angle
            angle_stats.append({
                'Video': video_name,
                'Max': np.max(angle),
                'Min': np.min(angle),
                'ROM': np.max(angle) - np.min(angle),
                'Mean': np.mean(angle),
                'Median': np.median(angle),
                'Std_Dev': np.std(angle),
                'Mode': stats.mode(angle, keepdims=True).mode[0]
            })
            
            # 3. Calculate Descriptive Statistics for Kinematics (Including Mode)
            kinematic_stats.append({
                'Video': video_name,
                # Velocity Stats
                'Vel_Max': np.max(vel), 'Vel_Min': np.min(vel), 
                'Vel_Mean': np.mean(vel), 'Vel_Median': np.median(vel), 
                'Vel_Std': np.std(vel), 'Vel_Mode': stats.mode(np.round(vel, 2), keepdims=True).mode[0],
                
                # Acceleration Stats
                'Acc_Max': np.max(acc), 'Acc_Min': np.min(acc), 
                'Acc_Mean': np.mean(acc), 'Acc_Median': np.median(acc), 
                'Acc_Std': np.std(acc), 'Acc_Mode': stats.mode(np.round(acc, 2), keepdims=True).mode[0],
                
                # Jerk Stats
                'Jerk_Max': np.max(jerk), 'Jerk_Min': np.min(jerk), 
                'Jerk_Mean': np.mean(jerk), 'Jerk_Median': np.median(jerk), 
                'Jerk_Std': np.std(jerk), 'Jerk_Mode': stats.mode(np.round(jerk, 2), keepdims=True).mode[0]
            })

        except Exception as e:
            print(f"Error processing {file}: {e}")

    if not time_wise_data:
        print("No data processed.")
        return

    # Combine all data
    df_master_time = pd.concat(time_wise_data, ignore_index=True)
    df_angle_summary = pd.DataFrame(angle_stats)
    df_kinematic_summary = pd.DataFrame(kinematic_stats)

    # --- SET DOWNLOAD PATH ---
    downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")
    output_file = os.path.join(downloads_path, "Biomechanics_Full_Report.xlsx")

    # --- SAVE TO EXCEL ---
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_master_time.to_excel(writer, sheet_name='TimeWise_Data', index=False)
        df_angle_summary.to_excel(writer, sheet_name='Angle_Descriptive_Stats', index=False)
        df_kinematic_summary.to_excel(writer, sheet_name='Kinematic_Stats', index=False)

    # --- PRINT PREVIEW ---
    print("\n--- Kinematic Statistics Preview ---")
    print(tabulate(df_kinematic_summary.head(10), headers='keys', tablefmt='fancy_grid', numalign="right", floatfmt=".4f"))

# --- RUN ---
LOCAL_PATH = r'C:\Users\PRASHANTH\Sports2D_v.0.8.27\main_angle_data'
process_biomechanics_data(LOCAL_PATH)


--- Kinematic Statistics Preview ---
╒════╤════════════╤═══════════╤═══════════╤════════════╤══════════════╤═══════════╤════════════╤═══════════╤════════════╤════════════╤══════════════╤═══════════╤════════════╤═════════════╤═════════════╤═════════════╤═══════════════╤════════════╤═════════════╕
│    │ Video      │   Vel_Max │   Vel_Min │   Vel_Mean │   Vel_Median │   Vel_Std │   Vel_Mode │   Acc_Max │    Acc_Min │   Acc_Mean │   Acc_Median │   Acc_Std │   Acc_Mode │    Jerk_Max │    Jerk_Min │   Jerk_Mean │   Jerk_Median │   Jerk_Std │   Jerk_Mode │
╞════╪════════════╪═══════════╪═══════════╪════════════╪══════════════╪═══════════╪════════════╪═══════════╪════════════╪════════════╪══════════════╪═══════════╪════════════╪═════════════╪═════════════╪═════════════╪═══════════════╪════════════╪═════════════╡
│  0 │ squat_10   │  172.9945 │ -235.1914 │     8.6092 │       8.4939 │   99.1767 │  -235.1900 │ 1870.8637 │ -1017.7701 │   -35.3287 │     -80.8149 │  449.3199 │ -1017.7700 │  18712.